# 24. LLM Evals Fundamentals

**Tier:** Evaluation & Production
**Estimated time:** 50 minutes
**Prerequisites:** 19, 23
**Priority:** 🔴 Crucial — "how do you know it works?" is the question that separates AI Engineers from demo-builders; every later notebook's quality gate (27, 27b, 33, P2, P4) builds on the harness you write here. *If skipped, revisit when:* immediately — there is no sensible skip path, this gates almost everything downstream.
**Source material:** Stanford Lecture 8 (LLM evaluation) — https://x.com/ajitcodes/status/2057043965317165490 ; @theahmadosman roadmap — https://x.com/theahmadosman/status/2062343535144436073

## What You'll Learn
- Why an eval is a *test suite for behavior*, not for code — and why "it looks good" doesn't scale
- Three scoring strategies: exact-match, rubric-based, and model-graded — and when each is appropriate
- pass@k and why it matters for anything with sampling variance
- A reusable error-analysis taxonomy for turning eval failures into fixes

## Why This Matters
Anyone can wire up a call to Claude and eyeball the output. What separates a professional AI Engineer is being able to answer, with evidence, "is this version actually better than the last one?" Every notebook after this one — production monitoring, agent evals, CI gates, both capstone RAG and multi-agent projects — assumes you already have an eval harness. This notebook builds the one you'll reuse everywhere else in the curriculum.


## What is an eval, really?

Think of unit tests for code: you give a function known inputs, assert on known outputs, and re-run automatically on every change. An **eval** is the same idea applied to model *behavior* instead of code — a fixed set of inputs (a "golden dataset"), a way to score each output, and a way to aggregate scores into one number you can track over time.

The reason this is harder than unit testing: LLM outputs aren't deterministic strings you can `assert ==` against. "Paris is the capital of France" and "The capital of France is Paris" are both correct, but a naive string match fails one of them. So evals need **scorers** — functions that grade an output against a reference, ranging from strict (exact match) to loose (a second LLM judging quality).

Three scorer families, roughly in order of cost and flexibility:

1. **Exact / structural match** — cheap, deterministic, but brittle. Good for classification, extraction, anything with one right answer.
2. **Rubric-based (rule/heuristic) scoring** — a checklist of *properties* the answer should have (contains X, doesn't contain Y, length in range). More forgiving than exact match, still deterministic.
3. **Model-graded ("LLM-as-judge")** — a second LLM call scores the output against criteria in natural language. Flexible and scales to subjective quality, but introduces judge bias (notebook 26 covers this in depth).

You'll build all three below, on the same golden dataset, so you can see the tradeoffs directly.

In [ ]:
import os, json
import numpy as np
import matplotlib.pyplot as plt

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

HAS_ANTHROPIC = bool(os.environ.get("ANTHROPIC_API_KEY"))
TEACH_MODEL = "claude-haiku-4-5-20251001"   # production default: claude-opus-4-8

if HAS_ANTHROPIC:
    import anthropic
    client = anthropic.Anthropic()
    print("Anthropic ready.")
else:
    client = None
    print("No ANTHROPIC_API_KEY — model-graded cells will be skipped.")

def ask(prompt, system="You are concise.", max_tokens=300, temperature=0.7):
    if not HAS_ANTHROPIC:
        return "[skipped: no ANTHROPIC_API_KEY]"
    try:
        msg = client.messages.create(model=TEACH_MODEL, max_tokens=max_tokens, system=system,
                                      temperature=temperature, messages=[{"role": "user", "content": prompt}])
        return msg.content[0].text
    except Exception as e:
        return f"[skipped: {type(e).__name__}: {str(e)[:120]}]"


## The golden dataset

A golden dataset is just a list of `(input, reference)` pairs with known-correct answers. Small (10-50 examples) is fine to start — the discipline of having *any* fixed dataset beats eyeballing outputs every time. We'll use a tiny geography/arithmetic set so scoring is unambiguous, then generate answers from two "model versions" — a `terse` prompt and a `verbose` prompt — to compare.

In [ ]:
GOLDEN = [
    {"q": "What is the capital of France?", "ref": "Paris"},
    {"q": "What is the capital of Japan?", "ref": "Tokyo"},
    {"q": "What is 12 + 7?", "ref": "19"},
    {"q": "What is the capital of Australia?", "ref": "Canberra"},
    {"q": "What is 9 * 6?", "ref": "54"},
    {"q": "What is the capital of Canada?", "ref": "Ottawa"},
]

def run_variant(system_prompt, label):
    outputs = []
    for item in GOLDEN:
        out = ask(item["q"], system=system_prompt, max_tokens=60, temperature=0.0)
        outputs.append({**item, "output": out, "variant": label})
    return outputs

TERSE_SYSTEM = "Answer with only the fact requested. No explanation, no extra words."
VERBOSE_SYSTEM = "You are a friendly tutor. Explain your reasoning before giving the answer."

terse_outputs = run_variant(TERSE_SYSTEM, "terse")
verbose_outputs = run_variant(VERBOSE_SYSTEM, "verbose")

for o in terse_outputs[:3]:
    print(f"Q: {o['q']}\n  terse -> {o['output']!r}")


## Scorer 1 — exact match (normalized)

The cheapest scorer: lowercase, strip punctuation, check if the reference string appears in the output. Fast and deterministic, but notice how it will unfairly penalize the verbose variant even when it gets the fact right — that's the brittleness called out above.

In [ ]:
import re

def normalize(s):
    return re.sub(r"[^a-z0-9]", "", s.lower())

def exact_match_score(output, reference):
    return 1.0 if normalize(reference) in normalize(output) else 0.0

def score_variant(outputs, scorer):
    scores = [scorer(o["output"], o["ref"]) for o in outputs]
    return scores

terse_exact = score_variant(terse_outputs, exact_match_score)
verbose_exact = score_variant(verbose_outputs, exact_match_score)

print(f"Terse exact-match accuracy:   {np.mean(terse_exact):.0%}")
print(f"Verbose exact-match accuracy: {np.mean(verbose_exact):.0%}")


## Scorer 2 — rubric-based (heuristic checklist)

A rubric scorer checks a checklist of *properties* instead of one exact string. Here: does the output contain the reference fact (loosely), and does it respect an expected length band? This is still fully deterministic — no second model call — but far less brittle than exact match.

In [ ]:
def rubric_score(output, reference, max_words=40):
    contains_fact = normalize(reference) in normalize(output)
    word_count = len(output.split())
    concise_enough = word_count <= max_words
    # Weighted checklist: correctness matters far more than length.
    return 0.8 * contains_fact + 0.2 * concise_enough

terse_rubric = score_variant(terse_outputs, rubric_score)
verbose_rubric = score_variant(verbose_outputs, rubric_score)

print(f"Terse rubric score:   {np.mean(terse_rubric):.2f}")
print(f"Verbose rubric score: {np.mean(verbose_rubric):.2f}")


## Scorer 3 — model-graded ("LLM-as-judge")

For anything subjective (tone, helpfulness, faithfulness), a second LLM call scores the output on a 0-1 scale against explicit criteria. This is the most flexible scorer and the one that scales best to open-ended tasks — notebook 26 covers judge bias and calibration in depth. Here we use a strict grading prompt that forces a single number, so we can parse it reliably.

In [ ]:
JUDGE_SYSTEM = (
    "You are a strict grader. Given a question, a reference answer, and a candidate answer, "
    "output ONLY a single number between 0 and 1 (e.g. 0.0, 0.5, 1.0) scoring whether the "
    "candidate answer contains the correct fact from the reference. No explanation."
)

def judge_score(question, reference, candidate):
    if not HAS_ANTHROPIC:
        return None
    prompt = f"Question: {question}\nReference: {reference}\nCandidate: {candidate}\nScore:"
    raw = ask(prompt, system=JUDGE_SYSTEM, max_tokens=10, temperature=0.0)
    try:
        return float(re.findall(r"[01](?:\.\d+)?", raw)[0])
    except (IndexError, ValueError):
        return None

if HAS_ANTHROPIC:
    terse_judge = [judge_score(o["q"], o["ref"], o["output"]) for o in terse_outputs]
    verbose_judge = [judge_score(o["q"], o["ref"], o["output"]) for o in verbose_outputs]
    terse_judge = [s for s in terse_judge if s is not None]
    verbose_judge = [s for s in verbose_judge if s is not None]
    print(f"Terse judge score:   {np.mean(terse_judge):.2f}" if terse_judge else "Terse judge score: n/a")
    print(f"Verbose judge score: {np.mean(verbose_judge):.2f}" if verbose_judge else "Verbose judge score: n/a")
else:
    terse_judge, verbose_judge = [], []
    print("[skipped: no ANTHROPIC_API_KEY]")


## pass@k — scoring under sampling variance

Many tasks (code generation, agent tool calls) have randomness: the same prompt can succeed once and fail the next. **pass@k** answers "if I sample k times, what's the probability at least one succeeds?" It's the standard metric whenever you can retry, and it's very different from single-sample accuracy — a low per-sample accuracy can still have a high pass@k if failures are independent.

In [ ]:
def pass_at_k(n_samples, n_correct, k):
    """Unbiased pass@k estimator (Codex/HumanEval formula)."""
    if n_samples - n_correct < k:
        return 1.0
    # 1 - P(all k draws come from the n_samples - n_correct incorrect ones).
    return 1.0 - np.prod([(n_samples - n_correct - i) / (n_samples - i) for i in range(k)])

# Simulate: a flaky task that succeeds independently 30% of the time per sample.
np.random.seed(0)
n_samples, p_success = 10, 0.3
successes = np.random.binomial(1, p_success, size=n_samples).sum()

for k in [1, 3, 5]:
    print(f"pass@{k} with {successes}/{n_samples} successes at p={p_success}: {pass_at_k(n_samples, successes, k):.2f}")


## Visualizing the three scorers side by side

The same two variants, scored three different ways, tell three different stories — which is exactly why picking the right scorer for the task matters more than picking the "best" model.

In [ ]:
%matplotlib inline
labels = ["exact match", "rubric", "judge"]
terse_means = [np.mean(terse_exact), np.mean(terse_rubric),
               np.mean(terse_judge) if terse_judge else 0]
verbose_means = [np.mean(verbose_exact), np.mean(verbose_rubric),
                 np.mean(verbose_judge) if verbose_judge else 0]

x = np.arange(len(labels))
width = 0.35
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(x - width/2, terse_means, width, label="terse prompt")
ax.bar(x + width/2, verbose_means, width, label="verbose prompt")
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_ylabel("score (0-1)")
ax.set_ylim(0, 1.05)
ax.set_title("Same outputs, three scorers: which one tells the truth?")
ax.legend()
plt.tight_layout()
plt.show()


*Exact match unfairly penalizes the verbose variant for phrasing; the rubric and judge scorers recover its actual correctness. No single scorer is "right" — pick the one that matches what you actually care about.*

## Error-analysis taxonomy

When an eval score drops, "why" matters more than "how much." A reusable taxonomy turns raw failures into an actionable fix list instead of a vague number:

- **Factual error** — the model got the fact wrong (real capability gap)
- **Format violation** — right fact, wrong shape (breaks downstream parsing)
- **Refusal / non-answer** — the model declined or hedged
- **Reference is wrong** — the golden dataset itself has a bug (happens more than people admit)
- **Scorer is wrong** — the output was fine but the scorer misjudged it (as seen above with exact match)

Classifying every failure into one of these before "fixing" anything prevents chasing model quality when the real bug is in your golden dataset or your scorer.

In [ ]:
def classify_failure(item, exact_score):
    if exact_score >= 1.0:
        return "pass"
    out_lower = item["output"].lower()
    if any(w in out_lower for w in ["cannot", "i don't know", "unable", "sorry"]):
        return "refusal"
    if len(item["output"].split()) > 40:
        return "format_violation (too verbose to be brittle exact-match)"
    return "factual_error_or_scorer_miss"

for o, s in zip(verbose_outputs, verbose_exact):
    tag = classify_failure(o, s)
    if tag != "pass":
        print(f"[{tag}] Q={o['q']!r} -> {o['output'][:60]!r}")


## Exercises

**Exercise 1 (Warm-up):** Add two new golden-dataset items of your own (any facts you can verify) and re-run all three scorers. Does the ranking between `terse` and `verbose` change?

**Exercise 2 (Apply):** Implement a fourth scorer, `contains_all_keywords(output, required_keywords: list[str])`, that passes only if every keyword in a list appears in the output. Compare it against `rubric_score` on the existing dataset.

**Exercise 3 (Extend):** This harness only measures single-turn Q&A. Sketch (in comments or a short function) how you'd adapt `run_variant` + a scorer to evaluate the multi-turn research agent from notebook 19 — what's the golden reference for an agent trajectory, and which scorer family fits best? (Notebook 27b builds this for real.)


In [ ]:
# Exercise 1: Warm-up
# Task: Add 2 golden-dataset items and re-run all three scorers on terse vs verbose.
# Hint: append to a copy of GOLDEN, don't mutate the original list in place.

# YOUR CODE HERE


# Exercise 2: Apply
# Task: Implement contains_all_keywords(output, required_keywords) and compare it to rubric_score.
# Hint: normalize() from earlier handles casing/punctuation for you.

# YOUR CODE HERE


# Exercise 3: Extend
# Task: Sketch how run_variant + a scorer would evaluate an agent trajectory (notebook 19),
# not just a single answer.
# Hint: the "reference" for a trajectory might be a set of required tool calls, not a string.

# YOUR CODE HERE


<details>
<summary>Click to reveal solutions</summary>

```python
# Exercise 1
extended_golden = GOLDEN + [
    {"q": "What is the capital of Germany?", "ref": "Berlin"},
    {"q": "What is 15 - 6?", "ref": "9"},
]
terse_ext = run_variant(TERSE_SYSTEM, "terse")  # re-run with extended_golden substituted for GOLDEN
verbose_ext = run_variant(VERBOSE_SYSTEM, "verbose")

# Exercise 2
def contains_all_keywords(output, required_keywords):
    norm_out = normalize(output)
    return 1.0 if all(normalize(k) in norm_out for k in required_keywords) else 0.0

kw_scores = [contains_all_keywords(o["output"], [o["ref"]]) for o in verbose_outputs]
print(np.mean(kw_scores), np.mean(verbose_rubric))

# Exercise 3
# A trajectory "reference" isn't a string — it's a set of expected properties:
#   - required_tools: {"web_search", "write_file"}   (did it call the tools it needed to?)
#   - max_steps: 6                                     (did it stay within budget?)
#   - final_answer_scorer: judge_score(...)             (is the synthesized answer correct?)
# The right scorer family is a *rubric* over the trajectory (tool set, step count) combined
# with a *model-graded* score on the final answer — exactly what notebook 27b builds.
```
</details>

## Key Takeaways
- An eval is a fixed golden dataset + a scorer + an aggregate number you track over time — the same discipline as unit tests, applied to model behavior.
- Exact match is cheap but brittle; rubric scoring adds tolerance without a second model call; model-graded judging is the most flexible but introduces bias (notebook 26).
- pass@k answers "if I can retry k times, what's my success probability?" — essential for anything with sampling variance, especially agents.
- When a score drops, classify the failure (factual error, format violation, refusal, bad reference, bad scorer) before "fixing" the model — often the bug is in the eval, not the model.
- This harness (golden dataset + `run_variant` + scorer) is reused, unmodified in spirit, by notebooks 26, 27, 27b, 33, and both capstones P2/P4.

## What's Next
Notebook 25 covers benchmark hygiene — how the golden dataset itself can quietly become contaminated and stop measuring what you think it measures.
